# `comprehensive_test.py` — Full Model Diagnostic

## Purpose
End-to-end diagnostic that exercises every major capability of the trained model:
1. **Per-aspect prediction** on 10 diverse review texts
2. **Mixed Sentiment Resolution (MSR)** — detects conflicting sentiments across aspects
3. **Attention-based XAI** — top-5 tokens driving each aspect prediction
4. **Integrated Gradients** (optional, requires `captum`)
5. **MSR Delta** — probability shift from temperature=1.0 to calibrated temperature

## Expected accuracy
| Threshold | Result |
|-----------|--------|
| ≥ 70% accuracy AND all MSR checks pass | **PASS** |
| ≥ 50% accuracy | **PARTIAL** — model needs more training |
| < 50% | **FAIL** |

> **Requires a trained checkpoint** at `ml-research/outputs/cosmetic_sentiment_v1/best_model.pt`.

In [ ]:
import sys, os, json, time
import numpy as np

# ── Path setup ────────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.path.abspath('')
PROJECT_ROOT  = NOTEBOOK_DIR
while not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')):
    parent = os.path.dirname(PROJECT_ROOT)
    if parent == PROJECT_ROOT: break
    PROJECT_ROOT = parent

INFERENCE_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'cosmetic_sentiment_v1', 'evaluation')
SRC_DIR       = os.path.join(PROJECT_ROOT, 'src')
for p in [PROJECT_ROOT, INFERENCE_DIR, SRC_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

CKPT = os.path.join(PROJECT_ROOT, 'outputs', 'cosmetic_sentiment_v1', 'best_model.pt')
print('Project root:', PROJECT_ROOT)
print('Checkpoint  :', CKPT)
print('Exists      :', os.path.exists(CKPT))

In [ ]:
# ── Test reviews ──────────────────────────────────────────────────────────────
REVIEWS = [
    ('strongly_pos',
     'This lipstick is absolutely amazing! The colour is gorgeous, it lasts all day, '
     'smells divine and the price is very affordable.'),
    ('strongly_neg',
     'Worst product ever. The colour is terrible, it fades in one hour, smells horrible '
     'like chemicals, and the packaging broke on arrival. Way overpriced.'),
    ('mixed_colour_pos_smell_neg',
     'The colour is beautiful but the smell is absolutely awful. '
     'It also fades too quickly and the price is too high.'),
    ('smell_neg_texture_pos',
     'Great colour and the texture is silky smooth, '
     'but the scent is absolutely disgusting and unbearable.'),
    ('colour_pos_only',
     'The colour is breathtaking - such a rich vibrant shade. Perfect pigmentation.'),
    ('neutral_generic',
     'It is an okay product. Nothing special but it does what it says on the box.'),
    ('mixed_price_pos_texture_neg',
     'Very affordable price and great value for money. '
     'However the texture feels rough and heavy on the skin.'),
    ('packing_neg',
     'Packaging was completely destroyed when it arrived. '
     'The box was crushed and the product was unusable.'),
    ('shipping_neg',
     'The product arrived two weeks late and was clearly mis-handled during delivery.'),
    ('long_review',
     'Using this moisturizer for three months now. Texture is silky smooth and absorbs '
     'well. Skin feels noticeably softer. Staying power is excellent throughout the day. '
     'However the scent is very strong, almost medicinal - quite unpleasant. Packaging '
     'cracked when dropped. Very expensive for what you get.'),
]

EXPECTED = {
    'strongly_pos'              : {'colour': 'positive', 'smell': 'positive', 'price': 'positive'},
    'strongly_neg'              : {'colour': 'negative', 'smell': 'negative', 'price': 'negative'},
    'mixed_colour_pos_smell_neg': {'colour': 'positive', 'smell': 'negative'},
    'smell_neg_texture_pos'     : {'texture': 'positive', 'smell': 'negative'},
    'colour_pos_only'           : {'colour': 'positive'},
    'mixed_price_pos_texture_neg': {'price': 'positive', 'texture': 'negative'},
    'packing_neg'               : {'packing': 'negative'},
    'shipping_neg'              : {'shipping': 'negative'},
}

MIXED_CASES = [
    {'text':  'The colour is beautiful but the smell is absolutely awful.',
     'checks': [('colour', 'positive'), ('smell', 'negative')],
     'label':  'colour=pos, smell=neg'},
    {'text':  'Great texture but way overpriced. The packaging is decent.',
     'checks': [('texture', 'positive'), ('price', 'negative')],
     'label':  'texture=pos, price=neg'},
]

print(f'[OK] {len(REVIEWS)} reviews, {len(MIXED_CASES)} MSR cases loaded')

## Run Diagnostic

Set `RUN_XAI = True` to also run Integrated Gradients and MSR Delta (slower).
Set `SAVE_CHARTS = True` to save XAI plots to `tests/xai_charts/`.

In [ ]:
RUN_XAI     = False  # Set True to enable IG + MSR delta tests
SAVE_CHARTS = False  # Set True to save XAI charts as PNGs

if not os.path.exists(CKPT):
    print(f'[SKIP] Checkpoint not found: {CKPT}')
    print('       Train first: python src/models/train.py --config configs/config.yaml')
else:
    from inference import SentimentPredictor

    print('='*72)
    print('  CLEARVIEW MODEL — COMPREHENSIVE DIAGNOSTIC TEST')
    print('='*72)

    predictor = SentimentPredictor(CKPT)
    aspects   = predictor.aspect_names
    print(f'Aspects : {aspects}')
    print(f'Device  : {predictor.device}')
    print()

    # ── Per-review predictions ────────────────────────────────────────────────
    total_expected = total_correct = 0
    all_confs, low_conf = [], 0

    for rev_name, text in REVIEWS:
        exp_map = EXPECTED.get(rev_name, {})
        print(f'[{rev_name}]')
        print(f'  "{text[:110]}{"..." if len(text)>110 else ""}')
        t0 = time.time()
        for asp in aspects:
            r    = predictor.predict(text, asp)
            prob = r['probabilities']
            conf = r['confidence']
            sent = r['sentiment']
            all_confs.append(conf)
            if conf < 0.40: low_conf += 1
            expected = exp_map.get(asp)
            if expected is not None:
                total_expected += 1
                ok = sent == expected
                total_correct += int(ok)
                marker = 'OK ' if ok else 'ERR'
                flag   = '' if ok else f'  <- expected {expected}'
            else:
                marker, flag = '   ', ''
            print(f'  [{marker}] {asp:<15} -> {sent:<9} conf={conf:.3f}  '
                  f'neg={prob["negative"]:.2f} neu={prob["neutral"]:.2f} pos={prob["positive"]:.2f}{flag}')
        print(f'  Inference: {(time.time()-t0)*1000:.0f}ms for {len(aspects)} aspects')
        print()

    # ── Summary stats ─────────────────────────────────────────────────────────
    avg_conf = float(np.mean(all_confs)) if all_confs else 0
    acc_pct  = 100 * total_correct / max(total_expected, 1)
    print('='*72)
    print('SUMMARY')
    print('='*72)
    print(f'  Accuracy   : {total_correct}/{total_expected}  ({acc_pct:.1f}%)')
    print(f'  Avg conf   : {avg_conf:.3f}')
    print(f'  Low conf   : {low_conf}/{len(all_confs)} (conf < 0.40)')

    # ── MSR ──────────────────────────────────────────────────────────────────
    print()
    print('[Mixed Sentiment Resolution Checks]')
    mixed_pass = 0
    for case in MIXED_CASES:
        ok_all, lines = True, []
        for asp, exp in case['checks']:
            r  = predictor.predict(case['text'], asp)
            ok = r['sentiment'] == exp
            ok_all = ok_all and ok
            lines.append(f"{asp}={r['sentiment']}(conf={r['confidence']:.2f})")
        status = 'PASS' if ok_all else 'FAIL'
        mixed_pass += int(ok_all)
        print(f'  [{status}] {case["label"]}')
        print(f'    Text : "{case["text"][:70]}"')
        print(f'    Preds: {", ".join(lines)}')
    print(f'  MSR: {mixed_pass}/{len(MIXED_CASES)} cases correctly resolved')

    # ── Attention XAI ─────────────────────────────────────────────────────────
    print()
    print('[Attention XAI Check]')
    xai_text = 'The colour is beautiful but the smell is absolutely awful.'
    for asp in ['colour', 'smell']:
        try:
            r = predictor.predict(xai_text, asp, return_attention=True)
            if 'attention' not in r:
                print(f'  [WARN] No attention for {asp!r}')
                continue
            top5 = sorted(zip(r['attention']['tokens'], r['attention']['weights']),
                          key=lambda x: x[1], reverse=True)[:5]
            print(f'  Top-5 tokens for {asp!r} (pred={r["sentiment"]})')
            for tok, wt in top5:
                print(f'    {tok:<20} {"|"+"#"*int(wt*40):<44} {wt:.4f}')
        except Exception as exc:
            print(f'  [ERROR] {asp}: {exc}')

    # ── Optional XAI ─────────────────────────────────────────────────────────
    if RUN_XAI:
        charts_dir = os.path.join(PROJECT_ROOT, 'tests', 'xai_charts')
        os.makedirs(charts_dir, exist_ok=True)
        print()
        print('[Integrated Gradients Check]')
        ig_text = 'The colour is amazing but the smell is quite off-putting.'
        try:
            ig_save = os.path.join(charts_dir, 'ig_colour.png') if SAVE_CHARTS else None
            predictor.explain_with_integrated_gradients(ig_text, 'colour', n_steps=30, top_k=10, save_path=ig_save)
            print('  [OK] Integrated Gradients completed')
        except ImportError:
            print('  [SKIP] captum not installed: pip install captum')
        except Exception as exc:
            print(f'  [ERROR] IG failed: {exc}')

        print()
        print('[MSR Delta Check]')
        msr_text = 'I love the colour of this lipstick but the smell is absolutely revolting.'
        try:
            msr_save = os.path.join(charts_dir, 'msr_colour.png') if SAVE_CHARTS else None
            predictor.explain_msr_delta(msr_text, 'colour', top_k=8, save_path=msr_save)
            print('  [OK] MSR Delta completed')
        except Exception as exc:
            print(f'  [ERROR] MSR Delta: {exc}')

    # ── Final verdict ─────────────────────────────────────────────────────────
    print()
    print('='*72)
    if acc_pct >= 70 and mixed_pass == len(MIXED_CASES):
        print('[RESULT] PASS — Model performing correctly')
    elif acc_pct >= 50:
        print('[RESULT] PARTIAL — Model needs more training')
    else:
        print('[RESULT] FAIL — Low accuracy or mixed sentiment unresolved')
    print('='*72)

    # ── Save JSON report ──────────────────────────────────────────────────────
    out_path = os.path.join(PROJECT_ROOT, 'tests', 'comprehensive_diagnostic.json')
    # (report dict not built above for brevity — just signal where it would go)
    print(f'  Report would be saved to: {out_path}')